In [ ]:
# Import and run the data generator
import os

# Ensure we're in the correct directory (where data.py and config should be)
if not os.getcwd().endswith('data\\data'):
    os.chdir('c:/Users/Hp/Desktop/data/data')

from data import Data

# Initialize data generator (cleanup happens automatically during __init__)
# data_gen = Data(seed=42, simbench_code='1-LV-rural2--1-no_sw')
data_gen = Data(seed=42, simbench_code='1-LV-urban6--2-no_sw')


# Generate data (config file is automatically loaded from initialization)
print("Starting data generation...\n")

filtered_dfs, time_range = data_gen.generate_consistent_copies()
print("\n✅ Data generation complete!")
print(f"Generated {len(filtered_dfs)} datasets")


🗑️ Cleaning up existing generated files...
  Deleted: load_actual.csv
  Deleted: load_forecast_da.csv
  Deleted: load_forecast_id.csv
  Deleted: res_actual.csv
  Deleted: res_forecast_da.csv
  Deleted: res_forecast_id.csv
  Deleted: storage_actual.csv
  Deleted: storage_forecast_da.csv
  Deleted: storage_forecast_id.csv
  Deleted: prices.csv
✓ Deleted 10 files

Starting data generation...



FileNotFoundError: [Errno 2] No such file or directory: 'data_config.json'

In [ ]:
# Display Load Actual data
print("📊 LOAD ACTUAL DATA (MW)")
print("="*80)
print(f"Shape: {filtered_dfs['load_actual'].shape}")
print(f"Columns: {len(filtered_dfs['load_actual'].columns)}")
print(f"\nFirst 5 rows:")
filtered_dfs['load_actual'].head()


In [ ]:
# Print load columns list (excluding datetime, sorted by unit number)
import re

def extract_unit_id(col_name):
    """Extract unit number for sorting"""
    match = re.search(r'Load\s+(\d+)', col_name)
    return int(match.group(1)) if match else 0

sorted([col for col in filtered_dfs['load_actual'].columns if col != 'datetime'], key=extract_unit_id)


In [ ]:
# Print RES columns list (excluding datetime, sorted by unit number)
import re

def extract_unit_id(col_name):
    """Extract unit number for sorting"""
    match = re.search(r'SGen\s+(\d+)', col_name)
    return int(match.group(1)) if match else 0

sorted([col for col in filtered_dfs['res_actual'].columns if col != 'datetime'], key=extract_unit_id)


In [ ]:
# Print storage columns list (excluding datetime, sorted by unit number)
import re

def extract_unit_id(col_name):
    """Extract unit number for sorting"""
    match = re.search(r'Storage\s+(\d+)', col_name)
    return int(match.group(1)) if match else 0

sorted([col for col in filtered_dfs['storage_actual'].columns if col != 'datetime'], key=extract_unit_id)


In [ ]:
# Display all units grouped by Bus, Type, and ordered by unit number
import re
import pandas as pd


def parse_column_name(col_name):
    """Parse column name to extract bus, type, and unit number"""
    match = re.match(r'LV\d+\.(\d+)\s+(Load|SGen|Storage)\s+(\d+)', col_name)
    if match:
        bus_num = int(match.group(1))
        unit_type = match.group(2)
        unit_num = int(match.group(3))
        return bus_num, unit_type, unit_num, col_name
    return None, None, None, None


def extract_units_from_dataframe(df, unit_type_name):
    """Extract units from a dataframe and return as DataFrame ordered by unit number"""
    units_data = []
    for col in df.columns:
        if col != 'datetime':
            bus, utype, unum, fullname = parse_column_name(col)
            if bus is not None:
                units_data.append({
                    'bus': bus,
                    'unit_type': unit_type_name,
                    'unit_number': unum,
                    'full_name': fullname
                })
    
    units_df = pd.DataFrame(units_data)
    if not units_df.empty:
        units_df = units_df.sort_values('unit_number').reset_index(drop=True)
    return units_df


# Step 1: Create dataframes for loads, res, and storage (ordered by unit number)
loads_df = extract_units_from_dataframe(filtered_dfs['load_actual'], 'Load')
res_df = extract_units_from_dataframe(filtered_dfs['res_actual'], 'SGen')
storages_df = extract_units_from_dataframe(filtered_dfs['storage_actual'], 'Storage')

# Step 2: Combine all units and group by bus and unit type, ordered by unit number
all_units_df = pd.concat([loads_df, res_df, storages_df], ignore_index=True)
all_units_df = all_units_df.sort_values(['bus', 'unit_type', 'unit_number']).reset_index(drop=True)

# Step 3: Display with proper cascading structure
print("📊 ALL UNITS GROUPED BY BUS AND TYPE")
print("="*80)
print(f"Total units: {len(all_units_df)}\n")

current_bus = None
current_type = None

for idx, row in all_units_df.iterrows():
    # New bus
    if row['bus'] != current_bus:
        current_bus = row['bus']
        current_type = None
        print(f"\n🔹 BUS {current_bus}")
    
    # New unit type within the same bus
    if row['unit_type'] != current_type:
        current_type = row['unit_type']
        print(f"   ├── {current_type}")
    
    # Display unit (check if it's the last unit in this type for proper tree formatting)
    next_idx = idx + 1
    is_last_in_type = (next_idx >= len(all_units_df) or 
                       all_units_df.iloc[next_idx]['bus'] != current_bus or 
                       all_units_df.iloc[next_idx]['unit_type'] != current_type)
    
    if is_last_in_type:
        print(f"   │   └── Unit {row['unit_number']}: {row['full_name']}")
    else:
        print(f"   │   ├── Unit {row['unit_number']}: {row['full_name']}")

print("\n" + "="*80)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


all_units_df

In [ ]:
# Draw SimBench network topology
import sys
sys.path.append('c:/Users/Hp/Desktop/data/simbench-develop')

import simbench as sb
import pandapower.plotting as plot
import matplotlib.pyplot as plt

# Load the SimBench network
net = sb.get_simbench_net('1-LV-urban6--2-no_sw')

# Create network visualization
print("🔌 SIMBENCH NETWORK TOPOLOGY")
print("="*80)
print(f"Network: 1-LV-urban6--2-no_sw")
print(f"Buses: {len(net.bus)}")
print(f"Lines: {len(net.line)}")
print(f"Loads: {len(net.load)}")
print(f"Generators (RES): {len(net.sgen)}")
print(f"Storage: {len(net.storage)}")
print(f"Transformers: {len(net.trafo)}")
print("\n")

# Plot the network
plt.figure(figsize=(16, 12))
plot.simple_plot(net, 
                 plot_loads=True, 
                 plot_sgens=True,
                 load_size=2.0,
                 sgen_size=2.0,
                 bus_size=1.5,
                 line_width=1.0,
                 respect_switches=True,
                 show_plot=False)

plt.title('SimBench Network: 1-LV-urban6--2-no_sw', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("="*80)